# Kalam on AMI: GPU benchmark

Runs Kalam's exact pipeline (`app.run_pipeline`) on the AMI Meeting Corpus with a GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

Protocol, so the result is publishable:
1. Compare configurations on the **dev** meetings only.
2. Pick the best configuration by dev score.
3. Run the **test** set once with that configuration and report it, alongside the earlier baselines.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

In [ ]:
!git clone -q https://github.com/faisalafieh/kalam
%cd kalam
!git log --oneline -1

### Install
torch 2.6 is the oldest CUDA build available for Colab's Python; the server pins 2.5.1 on Python 3.11.
Takes about 3 minutes. Ignore "dependency conflicts" warnings about gradio, numba, google-adk or diffusers:
those are Colab's own packages and are not used here.

In [ ]:
!pip install -q torch==2.6.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q fastapi==0.115.6 python-multipart==0.0.20 faster-whisper==1.1.1 "pyannote.audio[separation]==3.3.2" "huggingface_hub<1.0" "numpy<2.3" matplotlib -r eval/requirements.txt

In [ ]:
# faster-whisper (CTranslate2) needs to find the cuBLAS / cuDNN libraries that came with torch.
import os, glob, site
libs = sorted({os.path.dirname(p) for sp in site.getsitepackages()
               for p in glob.glob(f"{sp}/nvidia/*/lib/*.so*")})
os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
# torch 2.6 refuses to load pyannote 3.3 checkpoints by default (weights_only=True).
# These are the official pyannote models, so allow full loading.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.__version__.startswith("2.6"), (
    "Wrong torch version loaded. Runtime -> Disconnect and delete runtime, then Run all again.")
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."

### Hugging Face token
Needs a **read** token, with the terms accepted on all three model pages:
[pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1),
[pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0) and
[pyannote/speech-separation-ami-1.0](https://huggingface.co/pyannote/speech-separation-ami-1.0).

In [ ]:
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")
os.environ.update(DEVICE="cuda", COMPUTE_TYPE="float16", LANGUAGE="en", BEAM_SIZE="5")

## 1. Compare configurations on the dev set

In [ ]:
!python eval/prepare_ami.py --split dev

In [ ]:
import subprocess

def run(tag, manifest, **settings):
    """Run one configuration, streaming its output into the notebook."""
    env = {**os.environ, **{k: str(v) for k, v in settings.items()}}
    print(f"\n=== {tag}: {settings}", flush=True)
    proc = subprocess.Popen(
        ["python", "eval/evaluate.py", "--manifest", manifest, "--tag", tag,
         "--hardware", "Colab " + torch.cuda.get_device_name(0)],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, tag)

DEV = "eval/data/ami/manifest_dev_headset.jsonl"

# The configuration Kalam shipped with, before any fixes. Run on every split so
# the before/after comparison is like for like.
ORIGINAL = dict(WHISPER_MODEL="small", LANGUAGE="auto", BEAM_SIZE=1,
                CONDITION_ON_PREVIOUS="true", WORD_SPEAKERS="false", SEPARATE="false")

CONFIGS = {
    "dev-original":            ORIGINAL,
    "dev-small":               dict(WHISPER_MODEL="small"),
    "dev-large-v3-turbo":      dict(WHISPER_MODEL="large-v3-turbo"),
    "dev-large-v3":            dict(WHISPER_MODEL="large-v3"),
    # ablation: old segment-level speaker assignment, same model
    "dev-large-v3-segment":    dict(WHISPER_MODEL="large-v3", WORD_SPEAKERS="false"),
    # overlapping speech separated into one stream per speaker
    "dev-large-v3-separated":  dict(WHISPER_MODEL="large-v3", SEPARATE="true"),
    "dev-turbo-separated":     dict(WHISPER_MODEL="large-v3-turbo", SEPARATE="true"),
}
failed = []
for tag, settings in CONFIGS.items():
    try:
        run(tag, DEV, **settings)
    except subprocess.CalledProcessError:
        failed.append(tag)
        print(f"!!! {tag} failed (error above), continuing with the rest")
        if failed == list(CONFIGS)[:2]:
            raise RuntimeError("The first two configurations both failed, so the problem is the setup, "
                               "not a configuration. Send the error printed above.")
print("failed:", failed or "none")

In [ ]:
import json
dev = {t: json.load(open(f"eval/results/{t}.json"))["summary"] for t in CONFIGS if t not in failed}
for t, s in dev.items():
    print(f"{t:24s} WER {100*s['wer']:5.1f}%   cpWER {100*s['cpwer']:5.1f}%   DER {100*s['der']:5.1f}%   RTF {s['rtf']:.2f}")

BEST = min(dev, key=lambda t: dev[t]["cpwer"])   # selected on dev cpWER only
print("\nSelected on dev:", BEST, CONFIGS[BEST])

## 2. Final run on the full test set (once)

All 16 test meetings, about 9 hours of audio: once with the original configuration
(the baseline) and once with the configuration selected on dev. Roughly 2 hours on a T4.

In [ ]:
!python eval/prepare_ami.py --all
TEST = "eval/data/ami/manifest_headset.jsonl"
run("test-original", TEST, **ORIGINAL)       # the "before" number
run("test-final", TEST, **CONFIGS[BEST])     # the "after" number, chosen on dev

## 3. Download the results
Commit `eval/RESULTS.md` and the `eval/results/*.json` files to the repo (these contain no audio).

In [ ]:
!zip -q -j kalam_results.zip eval/RESULTS.md eval/results/*.json
from google.colab import files
files.download("kalam_results.zip")